# J2C-TST-EN: baseline training and generation

Runs Llama/Gemma with four-shot, LoRA, and prefix-tuning on the English corpus. The profile targets a 48 GB NVIDIA GPU with BF16/FlashAttention 2, 32 GB host RAM, and early stopping with patience 1. Installation, training, and generation are enabled; rerunning resumes Trainer checkpoints and continues generation from missing examples.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

def find_baselines_dir():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        for possible in (candidate, candidate / '5-baselines'):
            if (possible / 'train_adapter.py').is_file():
                return possible.resolve()
    raise FileNotFoundError('Cannot locate 5-baselines/train_adapter.py')

BASELINES_DIR = find_baselines_dir()
LANGUAGE = 'EN'
RUN_DIR = BASELINES_DIR / 'J2C-TST-EN'
OUTPUTS_ROOT = RUN_DIR / 'outputs'
PREDICTIONS_DIR = RUN_DIR / 'predictions'
INSTALL_DEPENDENCIES = True
EXECUTE_EXPENSIVE = True
FORCE_HF_LOGIN = False  # True only to replace the saved Hugging Face account/token
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
MODELS = ['llama', 'gemma']
METHODS = ['lora', 'prefix']
EARLY_STOPPING_PATIENCE = 1
PROFILE_VERSION = 'gpu48-v3-gemma-prefix-cache'
MODEL_PROFILES = {
    'llama': {'tag': 'gpu48_bf16_fa2_es1_b4', 'train_batch': 4, 'eval_batch': 8, 'grad_accum': 4, 'gen_batch': 4},
    'gemma': {'tag': 'gpu48_bf16_fa2_es1_gemma_b2', 'train_batch': 2, 'eval_batch': 4, 'grad_accum': 8, 'gen_batch': 2},
}
print('Baseline directory:', BASELINES_DIR)
print('Run directory:', RUN_DIR)
print('Profile version:', PROFILE_VERSION)
print('Gemma profile:', MODEL_PROFILES['gemma'])

Baseline directory: /home/user/J2C-TST/5-baselines
Run directory: /home/user/J2C-TST/5-baselines/J2C-TST-EN
Profile version: gpu48-v3-gemma-prefix-cache
Gemma profile: {'tag': 'gpu48_bf16_fa2_es1_gemma_b2', 'train_batch': 2, 'eval_batch': 4, 'grad_accum': 8, 'gen_batch': 2}


In [2]:
if INSTALL_DEPENDENCIES:
    command = [sys.executable, '-m', 'pip', 'install', '-r', str(BASELINES_DIR / 'requirements.txt')]
    print('>', subprocess.list2cmdline(command))
    subprocess.run(command, cwd=BASELINES_DIR, check=True)
    if sys.platform.startswith('linux'):
        flash_env = os.environ.copy()
        flash_env.setdefault('MAX_JOBS', '2')
        flash_env.setdefault('NVCC_THREADS', '2')
        command = [sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(BASELINES_DIR / 'requirements-flash-attn.txt')]
        print('>', subprocess.list2cmdline(command))
        subprocess.run(command, cwd=BASELINES_DIR, env=flash_env, check=True)
    elif EXECUTE_EXPENSIVE:
        raise RuntimeError('The FlashAttention 2 profile requires a Linux CUDA environment.')
    else:
        print('FlashAttention 2 installation skipped in a non-Linux dry run.')
    if EXECUTE_EXPENSIVE:
        check = "import psutil, torch, flash_attn; assert torch.cuda.is_available(), 'CUDA unavailable'; assert torch.cuda.is_bf16_supported(), 'The GPU does not support BF16'; vram=torch.cuda.get_device_properties(0).total_memory/1024**3; print(torch.cuda.get_device_name(0), f'VRAM: {vram:.1f} GiB', f'RAM: {psutil.virtual_memory().total/1024**3:.1f} GiB', 'BF16:', torch.cuda.is_bf16_supported(), 'flash-attn:', flash_attn.__version__)"
        subprocess.run([sys.executable, '-c', check], cwd=BASELINES_DIR, check=True)
else:
    print('Dependency installation skipped; set INSTALL_DEPENDENCIES=True in the configuration cell for a fresh environment.')

MODEL_REPOSITORIES = {'llama': 'meta-llama/Llama-3.2-1B', 'gemma': 'google/gemma-3-1b-it'}
if EXECUTE_EXPENSIVE:
    import inspect
    from huggingface_hub import get_token, hf_hub_download, login
    if FORCE_HF_LOGIN:
        os.environ.pop('HF_TOKEN', None)
        os.environ.pop('HUGGING_FACE_HUB_TOKEN', None)
    if FORCE_HF_LOGIN or get_token() is None:
        print('Hugging Face authentication is required. The token will not be stored in the notebook.')
        login_parameters = inspect.signature(login).parameters
        if 'skip_if_logged_in' in login_parameters:
            login(skip_if_logged_in=False)
        elif 'new_session' in login_parameters:
            login(new_session=True)
        else:
            login()
    for alias in MODELS:
        model_repository = MODEL_REPOSITORIES[alias]
        try:
            hf_hub_download(repo_id=model_repository, filename='config.json', token=get_token())
        except Exception as error:
            raise RuntimeError(
                f'Access denied to {model_repository}. Open https://huggingface.co/{model_repository}, '
                'accept its terms with the same account as the token, then rerun this cell. '
                'To switch accounts, set FORCE_HF_LOGIN=True and rerun the cell.'
            ) from error
        print('Verified Hugging Face access:', model_repository)

> /opt/ice4hpc/rel-24.04/conda/bin/python -m pip install -r /home/user/J2C-TST/5-baselines/requirements.txt
Defaulting to user installation because normal site-packages is not writeable
> /opt/ice4hpc/rel-24.04/conda/bin/python -m pip install --no-build-isolation -r /home/user/J2C-TST/5-baselines/requirements-flash-attn.txt
Defaulting to user installation because normal site-packages is not writeable
NVIDIA L40S VRAM: 44.5 GiB RAM: 755.1 GiB BF16: True flash-attn: 2.8.3


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

Verified Hugging Face access: meta-llama/Llama-3.2-1B


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

Verified Hugging Face access: google/gemma-3-1b-it


In [3]:
EXPECTED_PROFILE_VERSION = 'gpu48-v3-gemma-prefix-cache'
if globals().get('PROFILE_VERSION') != EXPECTED_PROFILE_VERSION:
    raise RuntimeError('Stale kernel configuration: restart the kernel and run every cell from the beginning.')
expected_gemma = {'tag': 'gpu48_bf16_fa2_es1_gemma_b2', 'train_batch': 2, 'eval_batch': 4, 'grad_accum': 8, 'gen_batch': 2}
if MODEL_PROFILES.get('gemma') != expected_gemma:
    raise RuntimeError(f'Unsafe Gemma profile: {MODEL_PROFILES.get("gemma")!r}')
compat_check = [sys.executable, '-c', 'from train_adapter import install_gemma3_prefix_cache_compat; print("Gemma prefix compatibility: available")']
subprocess.run(compat_check, cwd=BASELINES_DIR, check=True)

def run_script(arguments):
    command = [sys.executable, *map(str, arguments)]
    print('\n>', subprocess.list2cmdline(command))
    return subprocess.run(command, cwd=BASELINES_DIR, check=True)

for model_alias in MODELS:
    profile = MODEL_PROFILES[model_alias]
    for method in METHODS:
        output_dir = OUTPUTS_ROOT / profile['tag'] / model_alias / method
        args = ['train_adapter.py', '--language', LANGUAGE, '--model-id', model_alias,
                '--method', method, '--output-dir', output_dir,
                '--train-batch-size', profile['train_batch'], '--eval-batch-size', profile['eval_batch'],
                '--gradient-accumulation-steps', profile['grad_accum'],
                '--early-stopping-patience', EARLY_STOPPING_PATIENCE,
                '--early-stopping-threshold', '0.0',
                '--bf16', '--tf32', '--attention-implementation', 'auto',
                '--dataloader-num-workers', '2', '--group-by-length',
                '--optim', 'adamw_torch_fused', '--no-full-determinism', '--resume']
        if not EXECUTE_EXPENSIVE:
            args.extend(['--max-train-samples', '2', '--max-validation-samples', '2', '--dry-run'])
        run_script(args)

Gemma prefix compatibility: available

> /opt/ice4hpc/rel-24.04/conda/bin/python train_adapter.py --language EN --model-id llama --method lora --output-dir /home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_b4/llama/lora --train-batch-size 4 --eval-batch-size 8 --gradient-accumulation-steps 4 --early-stopping-patience 1 --early-stopping-threshold 0.0 --bf16 --tf32 --attention-implementation auto --dataloader-num-workers 2 --group-by-length --optim adamw_torch_fused --no-full-determinism --resume
{
  "language": "EN",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "model_id": "meta-llama/Llama-3.2-1B",
  "method": "lora",
  "train_rows": 973,
  "validation_rows": 139,
  "train_batch_size": 4,
  "eval_batch_size": 8,
  "gradient_accumulation_steps": 4,
  "effective_batch_size_single_gpu": 16,
  "bf16_required": true,
  "tf32": true,
  "attention_implementation_requested": "auto",
  "gradient_checkpointing": false,
  "full_determinism": false,
 

Tokenizing EN: 100%|██████████| 139/139 [00:00<00:00, 364.37 examples/s]
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


No complete Trainer checkpoint found; starting adapter training from the beginning


 33%|███▎      | 61/183 [02:04<03:04,  1.51s/it]

{'loss': 1.3329, 'grad_norm': 1.1856968402862549, 'learning_rate': 0.0001737704918032787, 'epoch': 0.41}
{'loss': 1.248, 'grad_norm': 1.0610096454620361, 'learning_rate': 0.000146448087431694, 'epoch': 0.82}


...

{'eval_loss': 1.9018040895462036, 'eval_runtime': 6.7788, 'eval_samples_per_second': 20.505, 'eval_steps_per_second': 5.163, 'epoch': 1.0}
{'loss': 1.8356, 'grad_norm': 0.38450226187705994, 'learning_rate': 0.0001191256830601093, 'epoch': 1.23}
{'loss': 1.688, 'grad_norm': 0.49325719475746155, 'learning_rate': 9.18032786885246e-05, 'epoch': 1.64}



 97%|█████████▋| 34/35 [00:06<00:00,  5.04it/s]
                                                 A
  0%|          | 0/35 [00:00<?, ?it/s]

{'eval_loss': 1.6449402570724487, 'eval_runtime': 6.7629, 'eval_samples_per_second': 20.553, 'eval_steps_per_second': 5.175, 'epoch': 2.0}
{'loss': 1.5068, 'grad_norm': 0.4587761163711548, 'learning_rate': 6.44808743169399e-05, 'epoch': 2.05}
{'loss': 1.5157, 'grad_norm': 0.5621786713600159, 'learning_rate': 3.71584699453552e-05, 'epoch': 2.46}
{'loss': 1.5642, 'grad_norm': 0.5159009695053101, 'learning_rate': 9.836065573770493e-06, 'epoch': 2.87}



 97%|█████████▋| 34/35 [00:06<00:00,  5.33it/s]
                                                 A
100%|██████████| 183/183 [07:34<00:00,  2.49s/it]A


{'eval_loss': 1.6189212799072266, 'eval_runtime': 6.7041, 'eval_samples_per_second': 20.733, 'eval_steps_per_second': 5.221, 'epoch': 3.0}
{'train_runtime': 454.8488, 'train_samples_per_second': 6.418, 'train_steps_per_second': 0.402, 'train_loss': 2.0871453102820556, 'epoch': 3.0}


In [4]:
for model_alias in MODELS:
    profile = MODEL_PROFILES[model_alias]
    few_shot_output = PREDICTIONS_DIR / f"{LANGUAGE}_{model_alias}_few_shot_{profile['tag']}.csv"
    args = ['generate_baseline.py', '--language', LANGUAGE, '--strategy', 'few_shot',
            '--model-id', model_alias, '--output', few_shot_output,
            '--batch-size', profile['gen_batch'], '--bf16',
            '--attention-implementation', 'auto', '--resume']
    if not EXECUTE_EXPENSIVE:
        args.extend(['--limit', '2', '--dry-run'])
    run_script(args)
    for method in METHODS:
        adapter = OUTPUTS_ROOT / profile['tag'] / model_alias / method
        output = PREDICTIONS_DIR / f"{LANGUAGE}_{model_alias}_{method}_{profile['tag']}.csv"
        args = ['generate_baseline.py', '--language', LANGUAGE, '--strategy', method,
                '--model-id', model_alias, '--adapter-path', adapter, '--output', output,
                '--batch-size', profile['gen_batch'], '--bf16',
                '--attention-implementation', 'auto', '--resume']
        if not EXECUTE_EXPENSIVE:
            args.extend(['--limit', '2', '--dry-run'])
        run_script(args)


> /opt/ice4hpc/rel-24.04/conda/bin/python generate_baseline.py --language EN --strategy few_shot --model-id llama --output /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv --batch-size 4 --bf16 --attention-implementation auto --resume
{
  "language": "EN",
  "strategy": "few_shot",
  "model_id": "meta-llama/Llama-3.2-1B",
  "adapter_path": null,
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "test_rows": 278,
  "demonstration_row_ids": [
    "901",
    "157",
    "33",
    "1070"
  ],
  "seed": 42,
  "batch_size": 4,
  "max_input_tokens": 16384,
  "max_new_tokens": 256,
  "demo_source_max_chars": null,
  "decoding": "greedy",
  "temperature": null,
  "load_in_4bit": false,
  "attention_implementation_requested": "auto",
  "bf16_required": true,
  "resume_requested": true,
  "output": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv"
}


Generating:   0%|          | 0/70 [00:00<?, ?it/s]You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Generating: 100%|██████████| 70/70 [05:49<00:00,  4.99s/it]


Wrote 278 predictions to /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_few_shot_gpu48_bf16_fa2_es1_b4.csv

> /opt/ice4hpc/rel-24.04/conda/bin/python generate_baseline.py --language EN --strategy lora --model-id llama --adapter-path /home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_b4/llama/lora --output /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_lora_gpu48_bf16_fa2_es1_b4.csv --batch-size 4 --bf16 --attention-implementation auto --resume
{
  "language": "EN",
  "strategy": "lora",
  "model_id": "meta-llama/Llama-3.2-1B",
  "adapter_path": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_b4/llama/lora",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "test_rows": 278,
  "demonstration_row_ids": [],
  "seed": 42,
  "batch_size": 4,
  "max_input_tokens": 16384,
  "max_new_tokens": 256,
  "demo_source_max_chars": null,
  "decoding": "greedy",
  "temperature": null,
  "load_in_4bit": false,

Generating:   0%|          | 0/70 [00:00<?, ?it/s]You're using a PreTrainedTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Generating: 100%|██████████| 70/70 [01:45<00:00,  1.50s/it]


Wrote 278 predictions to /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_lora_gpu48_bf16_fa2_es1_b4.csv

> /opt/ice4hpc/rel-24.04/conda/bin/python generate_baseline.py --language EN --strategy prefix --model-id llama --adapter-path /home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_b4/llama/prefix --output /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_llama_prefix_gpu48_bf16_fa2_es1_b4.csv --batch-size 4 --bf16 --attention-implementation auto --resume
{
  "language": "EN",
  "strategy": "prefix",
  "model_id": "meta-llama/Llama-3.2-1B",
  "adapter_path": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_b4/llama/prefix",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "test_rows": 278,
  "demonstration_row_ids": [],
  "seed": 42,
  "batch_size": 4,
  "max_input_tokens": 16384,
  "max_new_tokens": 256,
  "demo_source_max_chars": null,
  "decoding": "greedy",
  "temperature": null,
  "load_in_4bit": 

...

Wrote 278 predictions to /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_few_shot_gpu48_bf16_fa2_es1_gemma_b2.csv

> /opt/ice4hpc/rel-24.04/conda/bin/python generate_baseline.py --language EN --strategy lora --model-id gemma --adapter-path /home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_gemma_b2/gemma/lora --output /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2.csv --batch-size 2 --bf16 --attention-implementation auto --resume
{
  "language": "EN",
  "strategy": "lora",
  "model_id": "google/gemma-3-1b-it",
  "adapter_path": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_gemma_b2/gemma/lora",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "test_rows": 278,
  "demonstration_row_ids": [],
  "seed": 42,
  "batch_size": 2,
  "max_input_tokens": 16384,
  "max_new_tokens": 256,
  "demo_source_max_chars": null,
  "decoding": "greedy",
  "temperature": null,
  "

Generating:   0%|          | 0/139 [00:00<?, ?it/s]You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Generating: 100%|██████████| 139/139 [04:07<00:00,  1.78s/it]


Wrote 278 predictions to /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_lora_gpu48_bf16_fa2_es1_gemma_b2.csv

> /opt/ice4hpc/rel-24.04/conda/bin/python generate_baseline.py --language EN --strategy prefix --model-id gemma --adapter-path /home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_gemma_b2/gemma/prefix --output /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_prefix_gpu48_bf16_fa2_es1_gemma_b2.csv --batch-size 2 --bf16 --attention-implementation auto --resume
{
  "language": "EN",
  "strategy": "prefix",
  "model_id": "google/gemma-3-1b-it",
  "adapter_path": "/home/user/J2C-TST/5-baselines/J2C-TST-EN/outputs/gpu48_bf16_fa2_es1_gemma_b2/gemma/prefix",
  "corpus": "/home/user/J2C-TST/4-official_corpora/EN/J2C_TST_EN.csv",
  "test_rows": 278,
  "demonstration_row_ids": [],
  "seed": 42,
  "batch_size": 2,
  "max_input_tokens": 16384,
  "max_new_tokens": 256,
  "demo_source_max_chars": null,
  "decoding": "greedy",
  "temperature": nul

Generating:   0%|          | 0/139 [00:00<?, ?it/s]You're using a GemmaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Generating: 100%|██████████| 139/139 [06:43<00:00,  2.90s/it]


Wrote 278 predictions to /home/user/J2C-TST/5-baselines/J2C-TST-EN/predictions/EN_gemma_prefix_gpu48_bf16_fa2_es1_gemma_b2.csv


The six complete English prediction files are written to `J2C-TST-EN/predictions/` and can be validated by the English notebook in `6-evaluation/J2C-TST-EN`.